# Step 3: Poll every N seconds

![Step 3 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/03-step.png)

## What you're building today

Right now we grab ONE frame. That's a snapshot, not a watcher.

Today we make a **loop** — grab a frame, wait N seconds, grab another, wait again, forever (or until we tell it to stop). Each frame gets a timestamped filename so we know when it happened.

By the end, you should see jpgs piling up in `data/snapshots/`.

## Step 3.1 — What is polling?

**Polling** means: ask the camera for a frame every few seconds, like checking the mailbox at regular times.

Why not just open the stream and watch continuously? Two reasons:

1. **Birds don't move every second.** Polling every 5-10 seconds catches them just fine.
2. **Power.** A continuous stream keeps the network busy. Polling is gentler.

The right `N` depends on how busy your feeder is. We'll start with 5 seconds and you can tune later.

In [ ]:
# Reuse the grab-one-frame logic from notebook 02
import requests
import os
from datetime import datetime
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output

RUNNING_IN_COLAB = "COLAB_GPU" in os.environ
if RUNNING_IN_COLAB:
    SOURCE = "https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/data/samples/sample-bird.jpg"
else:
    PHONE_IP = "192.168.1.42"
    SOURCE = f"http://{PHONE_IP}:8080/photo.jpg"

Path("data/snapshots").mkdir(parents=True, exist_ok=True)
print("OK")

## Step 3.2 — Wrap the grab as a function

We did this once by hand. Now we'll wrap it so we can call it over and over. This is what a function does — it's a name for a recipe.

In [ ]:
def grab_one_frame(source_url: str) -> Path:
    """Grab one frame from the camera and save with a timestamped filename."""
    response = requests.get(source_url, timeout=10)
    response.raise_for_status()

    # 2026-07-07_18-55-23.jpg — sortable by name
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    out_path = Path(f"data/snapshots/{ts}.jpg")
    out_path.write_bytes(response.content)
    return out_path

# Try it once
saved = grab_one_frame(SOURCE)
print(f"Saved: {saved}")

## Step 3.3 — The polling loop

A loop repeats code. We want: grab → wait → grab → wait → ...

Two important details:

- **`time.sleep(seconds)`** makes the loop pause. Without it, the loop would run as fast as the network can carry frames — way too many.
- **A way to stop** — `KeyboardInterrupt` is what happens when you press the Stop button in Colab or Ctrl+C on your laptop. The loop catches it and exits cleanly.

We'll also limit to 5 frames so this notebook doesn't run forever.

In [ ]:
import time

POLL_INTERVAL_SEC = 5     # wait 5s between frames
MAX_FRAMES = 5            # stop after 5 frames (so this cell ends)

saved_paths = []
try:
    for i in range(MAX_FRAMES):
        path = grab_one_frame(SOURCE)
        saved_paths.append(path)
        print(f"[{i+1}/{MAX_FRAMES}] saved {path.name}")
        if i < MAX_FRAMES - 1:
            time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print("Stopped by user")

print(f"\nDone. {len(saved_paths)} frames captured.")

## Step 3.4 — Look at what we collected

Five timestamped jpgs. Sortable by filename. That's a real log.

In [ ]:
# List everything we just saved
snapshots = sorted(Path("data/snapshots").glob("*.jpg"))
print(f"Total snapshots: {len(snapshots)}")
for p in snapshots[-5:]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name}  ({size_kb:.1f} KB)")

# Show the last one
if snapshots:
    display(Image.open(snapshots[-1]))

## Done?

If you see 5 (or however many you set) jpgs in the file list, you're polling. 🎉

Try changing `MAX_FRAMES` to 20 and `POLL_INTERVAL_SEC` to 2 — does it work faster? Same code, different rhythm.

## What's next

Issue **#4: Detect (yes/no)**. We've been saving every frame, including empty ones. Next we teach the computer to ask: "is there actually a bird in this picture?" 🐦